In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
from sklearn.model_selection import StratifiedGroupKFold

def stratified_group_split(df, label_col='tb_status', group_col='participant', test_size=0.2, random_state=42):
    sgkf = StratifiedGroupKFold(n_splits=int(1/test_size), shuffle=True, random_state=random_state)
    for train_idx, val_idx in sgkf.split(df, df[label_col], df[group_col]):
        df_train = df.iloc[train_idx].reset_index(drop=True)
        df_val = df.iloc[val_idx].reset_index(drop=True)
        return df_train, df_val

In [ ]:
df_longi = pd.read_csv('/run/media/fourier/Data1/Pras/DatabaseLLM/coda/longitudinal_original.csv')
df_solic = pd.read_csv('/run/media/fourier/Data1/Pras/DatabaseLLM/coda/solicited_original.csv')
#df_solic['tb_status'] = df_solic['tb_status'].map({0: 3, 1: 4})

df_longi_train, df_longi_val = stratified_group_split(df_longi)
df_solic_train, df_solic_val = stratified_group_split(df_solic)

df_train = pd.concat([df_longi_train, df_solic_train], ignore_index=True)
df_val = pd.concat([df_longi_val, df_solic_val], ignore_index=True)

df_train = df_train.sample(frac=1, random_state=42).reset_index(drop=True)
df_val = df_val.sample(frac=1, random_state=42).reset_index(drop=True)


In [ ]:
import matplotlib.pyplot as plt

df_val['participant'].value_counts().plot(kind='hist', bins=20, edgecolor='black')
plt.xlabel('Count per participant')
plt.ylabel('Frequency')
plt.title('Distribution of Samples per Participant')
plt.show()


In [ ]:
df_longi['participant'].value_counts()

In [ ]:
assert not set(df_longi_train['participant']) & set(df_longi_val['participant'])
assert not set(df_soliic_train['participant']) & set(df_soliic_val['participant'])

print(df_soliic_train['tb_status'].value_counts(normalize=True))
print(df_soliic_val['tb_status'].value_counts(normalize=True))

In [ ]:
df_soliic_val

In [ ]:


participant_col = "participant"
label_col = "tb_status"

test_participants_df = (
    df_test.groupby(participant_col)[label_col]
    .agg(lambda x: x.mode()[0]) 
    .reset_index()
)

val_participants, _ = train_test_split(
    test_participants_df[participant_col],
    test_size=0.8,
    stratify=test_participants_df[label_col],
    random_state=42
)

df_val = df_train[df_train[participant_col].isin(val_participants)]
df_train = df_train[~df_train[participant_col].isin(val_participants)]
df_test = pd.concat([df_val, df_test], ignore_index=True)

cols = ["path_file", "tb_status"]
df_train = df_train[cols]
df_test = df_test[cols]

counts = df_train['tb_status'].value_counts()
min_class = counts.idxmin()
max_class = counts.idxmax()
df_min = df_train[df_train.tb_status == min_class]
df_max = df_train[df_train.tb_status == max_class]
df_max_down = resample(df_max,
                       replace=False,
                       n_samples=len(df_min),
                       random_state=42)
df_train = pd.concat([df_min, df_max_down]).sample(frac=1, random_state=42).reset_index(drop=True)

counts = df_test['tb_status'].value_counts()
min_class = counts.idxmin()
max_class = counts.idxmax()
df_min = df_test[df_test.tb_status == min_class]
df_max = df_test[df_test.tb_status == max_class]
df_max_down = resample(df_max,
                       replace=False,
                       n_samples=len(df_min),
                       random_state=42)
df_test = pd.concat([df_min, df_max_down]).sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
df_test['tb_status'].value_counts()